# Entrenamiento de los cultivos principales

Selecciona cultivos por superficie cosechada con cobertura suficiente y entrena un modelo equivalente a `v1` para cada uno usando clima, suelo e historial productivo calculado solo con anos anteriores. Esta etapa construye los modelos candidatos para la futura matriz municipio-cultivo; todavia no realiza la redistribucion.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.base import clone

from proyecto_agronomia.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from proyecto_agronomia.modeling.cultivos import (
    VARIABLES_CATEGORICAS, VARIABLES_NUMERICAS, catalogar_cultivos,
    cargar_registros_cultivos, preparar_cultivo, seleccionar_principales, separar_xy,
)
from proyecto_agronomia.modeling.registry import guardar_modelo, ruta_version_modelo
from proyecto_agronomia.modeling.train import crear_particiones, seleccionar_modelo
from proyecto_agronomia.modeling.train_original import comparar_modelos_originales

2026-11-16 19:16:32.171 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: B:\MCD2\JW\agro\proyecto_agronomia


In [2]:
RUTA_DATOS = PROCESSED_DATA_DIR / 'agricultura_clima_edafologia_historial_2003_2025.csv'
CANTIDAD_CULTIVOS = 385  # maximo; los filtros determinan cuantos son elegibles
UNIDAD = 'Tonelada'
MINIMO_ANIOS = 15
MINIMO_MUNICIPIOS = 100
ANIO_INICIO_PRUEBA = 2022
PROPORCION_MUNICIPIOS_RESERVADOS = 0.20
SEMILLA = 42
VERSION = 'v1_principales'
ENTRENAR = True

if not RUTA_DATOS.is_file():
    raise FileNotFoundError(f'No se encontro el dataset de entrenamiento: {RUTA_DATOS}')
print(f'Dataset de entrenamiento: {RUTA_DATOS}')

Dataset de entrenamiento: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_edafologia_historial_2003_2025.csv


## Catálogo y selección transparente

In [3]:
catalogo = catalogar_cultivos(ruta=RUTA_DATOS, unidad=UNIDAD)
principales = seleccionar_principales(
    catalogo, cantidad=CANTIDAD_CULTIVOS, minimo_anios=MINIMO_ANIOS,
    minimo_municipios=MINIMO_MUNICIPIOS,
)
principales.insert(0, 'posicion', range(1, len(principales) + 1))
display(principales)
assert 0 < len(principales) <= CANTIDAD_CULTIVOS
assert principales['anios'].ge(MINIMO_ANIOS).all()
assert principales['municipios'].ge(MINIMO_MUNICIPIOS).all()
print(f'Cultivos elegibles: {len(principales):,} de hasta {CANTIDAD_CULTIVOS:,} solicitados')

,posicion,Nomcultivo,superficie_cosechada,registros,anios,municipios
0,1,Maíz grano,1.609257e+08,106329,23,2432
1,2,Pastos y praderas,5.006403e+07,29090,21,1132
2,3,Sorgo grano,3.649216e+07,23562,23,832
3,4,Frijol,3.418383e+07,65956,23,2145
4,5,Caña de azúcar,1.702829e+07,7801,23,329
...,...,...,...,...,...,...
84,85,Rábano,5.449668e+04,2619,21,159
85,86,Zempoalxochitl,3.527392e+04,1721,23,143
86,87,Mamey,2.848560e+04,1769,21,119
87,88,Pitahaya,1.848369e+04,1104,21,109


Cultivos elegibles: 89 de hasta 385 solicitados


## Cargar una sola vez los registros seleccionados

La lectura por bloques evita cargar los 825 mil registros completos y evita releer el archivo para cada cultivo.

In [4]:
nombres_cultivos = principales['Nomcultivo'].tolist()
datos_principales = cargar_registros_cultivos(
    nombres_cultivos, ruta=RUTA_DATOS, unidad=UNIDAD,
)
display(datos_principales.groupby('Nomcultivo').agg(
    registros=('cvegeo', 'size'), municipios=('cvegeo', 'nunique'),
    anio_inicial=('Anio', 'min'), anio_final=('Anio', 'max'),
).sort_values('registros', ascending=False))

,registros,municipios,anio_inicial,anio_final
Nomcultivo,,,,
Maíz grano,106329,2432,2003,2025
Frijol,65956,2145,2003,2025
Avena forrajera en verde,30772,1042,2003,2025
Pastos y praderas,29090,1132,2003,2025
Tomate rojo,25708,1319,2003,2025
...,...,...,...,...
Pitahaya,1104,109,2003,2025
Canola,876,216,2003,2025
Trigo forrajero verde,843,150,2003,2025


## Entrenamiento reanudable

Cada modelo terminado se guarda inmediatamente. Si se interrumpe la ejecución, al correr de nuevo se omiten los artefactos ya existentes.

In [5]:
resumen_modelos = []
if ENTRENAR:
    for numero, cultivo in enumerate(nombres_cultivos, start=1):
        destino = ruta_version_modelo(cultivo, VERSION)
        if (destino / 'model.joblib').is_file():
            print(f'[{numero}/{len(nombres_cultivos)}] Omitido, ya existe: {cultivo}')
            continue
        print(f'[{numero}/{len(nombres_cultivos)}] Entrenando: {cultivo}')
        datos = preparar_cultivo(datos_principales, cultivo)
        particiones = crear_particiones(
            datos, anio_prueba=ANIO_INICIO_PRUEBA,
            proporcion_municipios=PROPORCION_MUNICIPIOS_RESERVADOS, semilla=SEMILLA,
        )
        entrenados, metricas = comparar_modelos_originales(datos, particiones, SEMILLA)
        ganador = seleccionar_modelo(metricas)
        X, y = separar_xy(datos)
        modelo_final = clone(entrenados[ganador]).fit(X, y)
        guardar_modelo(
            modelo=modelo_final, cultivo=cultivo, version=VERSION,
            metadata={
                'algoritmo': ganador, 'unidad': UNIDAD,
                'periodo_datos': f"{datos['Anio'].min()}-{datos['Anio'].max()}",
                'inicio_prueba': ANIO_INICIO_PRUEBA,
                'proporcion_municipios_reservados': PROPORCION_MUNICIPIOS_RESERVADOS,
                'semilla': SEMILLA, 'registros': len(datos),
                'municipios': datos['cvegeo'].nunique(), 'objetivo': 'Rendimiento',
                'seleccion': 'cultivos principales por superficie cosechada',
            },
            metricas={'comparacion_modelos': metricas.to_dict(orient='records')},
            variables=[*VARIABLES_NUMERICAS, *VARIABLES_CATEGORICAS],
        )
        mejores = metricas.loc[metricas['modelo'].eq(ganador)]
        for fila in mejores.to_dict(orient='records'):
            resumen_modelos.append({'cultivo': cultivo, 'ganador': ganador, **fila})
        print(f'  Guardado {ganador} en {destino}')
else:
    print('Entrenamiento omitido.')

[1/89] Entrenando: Maíz grano
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\maiz_grano\v1_principales
[2/89] Entrenando: Pastos y praderas
  Guardado hist_gradient_boosting en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\pastos_y_praderas\v1_principales
[3/89] Entrenando: Sorgo grano
  Guardado hist_gradient_boosting en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\sorgo_grano\v1_principales
[4/89] Entrenando: Frijol
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\frijol\v1_principales
[5/89] Entrenando: Caña de azúcar
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\cana_de_azucar\v1_principales
[6/89] Entrenando: Café cereza
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\cafe_cereza\v1_principales
[7/89] Entrenando: Trigo grano
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\trigo_grano\v1_principa

b:\MCD2\JW\agro\proyecto_agronomia\.venv\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
b:\MCD2\JW\agro\proyecto_agronomia\.venv\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
b:\MCD2\JW\agro\proyecto_agronomia\.venv\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


  Guardado hist_gradient_boosting en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\hortalizas\v1_principales
[85/89] Entrenando: Rábano
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\rabano\v1_principales
[86/89] Entrenando: Zempoalxochitl
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\zempoalxochitl\v1_principales
[87/89] Entrenando: Mamey
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\mamey\v1_principales
[88/89] Entrenando: Pitahaya
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\pitahaya\v1_principales
[89/89] Entrenando: Granada
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\granada\v1_principales


## Resumen de modelos entrenados en esta ejecución

In [6]:
df_resumen_modelos = pd.DataFrame(resumen_modelos)
if not df_resumen_modelos.empty:
    display(df_resumen_modelos.sort_values(['particion', 'rmse']))
    destino_resumen = INTERIM_DATA_DIR / 'modeling' / 'cultivos_principales_metricas.csv'
    destino_resumen.parent.mkdir(parents=True, exist_ok=True)
    df_resumen_modelos.to_csv(destino_resumen, index=False)
    print(f'Resumen guardado: {destino_resumen}')
else:
    print('No hubo modelos nuevos; los existentes fueron respetados.')

,cultivo,ganador,modelo,particion,registros,mae,rmse,r2,sesgo
103,Jamaica,random_forest,random_forest,prueba_espaciotemporal,69,0.058946,0.075887,0.852784,0.019813
7,Frijol,random_forest,random_forest,prueba_espaciotemporal,2140,0.106244,0.178545,0.859915,-0.013966
47,Ajonjolí,random_forest,random_forest,prueba_espaciotemporal,154,0.099174,0.195492,0.428490,-0.037388
37,Garbanzo grano,random_forest,random_forest,prueba_espaciotemporal,184,0.182375,0.260114,0.799435,0.017602
51,Chile seco,random_forest,random_forest,prueba_espaciotemporal,117,0.161964,0.268932,0.929045,-0.049915
...,...,...,...,...,...,...,...,...,...
150,Sábila,random_forest,random_forest,prueba_temporal,126,9.358558,13.911766,0.645000,-0.635548
94,Lechuga,random_forest,random_forest,prueba_temporal,815,3.113741,16.573557,0.459729,-1.561386
100,Pepino,random_forest,random_forest,prueba_temporal,1475,9.115200,18.621596,0.867026,-1.196351
58,Tomate rojo,random_forest,random_forest,prueba_temporal,3849,10.934308,24.429182,0.923757,-2.732819


Resumen guardado: B:\MCD2\JW\agro\proyecto_agronomia\data\interim\modeling\cultivos_principales_metricas.csv
